In [ ]:
!pip install langchain-pinecone langchain-huggingface pinecone-client langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.2

In [ ]:
import os
# --- 1. CONFIGURATION ---
os.environ['PINECONE_API_KEY'] = '######'

# declare simpel Q&A data

In [ ]:
import os
from langchain_core.documents import Document
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings



PINECONE_INDEX_NAME = "medical-sport-assistant"
TARGET_NAMESPACE = "agent_guide_Q&A"

# --- 2. DECLARE THE Q&A DATA ---
# This is your official website FAQ. You can easily add more dictionaries to this list.
faq_data = [
    {
        "question": "What is the Titanium Trio Clinical Assistant?",
        "answer": "The Titanium Trio Clinical Assistant is an advanced multimodal AI platform designed to provide real-time sports performance analysis and medical rehabilitation guidance using computer vision and verified knowledge databases.",
        "category": "General Info"
    },
    {
        "question": "How does the computer vision form check work?",
        "answer": "Users can upload video clips of their exercises (like squats, bench presses, or football passes). The AI tracks your joint kinematics, calculates angles in real-time, and provides immediate feedback on your posture, depth, and potential injury risks.",
        "category": "Features"
    },
    {
        "question": "Are the medical and rehabilitation suggestions safe?",
        "answer": "Our AI pulls its advice strictly from a verified database of medical, rehabilitation, and sports science literature. However, it is an assistant, not a doctor. Always consult a healthcare professional before starting a new rehab program.",
        "category": "Safety & Medical"
    },
    {
        "question": "How is my personal data and video stored?",
        "answer": "Your data is processed securely through a Medallion Architecture pipeline. We use strict data isolation to ensure your personal videos and medical queries remain private and are only accessible by you.",
        "category": "Privacy & Architecture"
    },
    {
        "question": "What exercises does the vision model currently support?",
        "answer": "The system currently supports a wide range of movements including squats, deadlifts, bench presses, bicep curls, high knees, butt kicks, and specialized football performance tracking.",
        "category": "Features"
    }
]

# --- 3. CONVERT TO LANGCHAIN DOCUMENTS ---
print(f"📝 Formatting {len(faq_data)} Q&A pairs into documents...")
faq_documents = []

for item in faq_data:
    # We combine the Q and A into the main content so the embedding captures both
    combined_content = f"Question: {item['question']}\nAnswer: {item['answer']}"

    # Create the LangChain Document with metadata
    doc = Document(
        page_content=combined_content,
        metadata={
            "source": "website_faq_database",
            "category": item['category'],
            "type": "qa_pair"
        }
    )
    faq_documents.append(doc)



📝 Formatting 5 Q&A pairs into documents...


# conect and upload the data to pinecone

In [ ]:
# --- 4. INITIALIZE BGE LARGE EMBEDDINGS ---
print("🧠 Loading BGE-Large Embedding Model...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda'}, # Change to 'cpu' if you are not using the Colab T4 GPU
    encode_kwargs={'normalize_embeddings': True}
)

# --- 5. THE UPLOAD (UPSERT) ---
print(f"\n🚀 Starting upload to index '{PINECONE_INDEX_NAME}' under namespace '{TARGET_NAMESPACE}'...")

vectorstore = PineconeVectorStore.from_documents(
    documents=faq_documents,
    embedding=embeddings,
    index_name=PINECONE_INDEX_NAME,
    namespace=TARGET_NAMESPACE
)
print("✅ Upload complete!")

# --- 6. VERIFICATION ---
print("\n📊 --- PINECONE INDEX ANALYSIS ---")
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index = pc.Index(PINECONE_INDEX_NAME)
stats = index.describe_index_stats()

namespaces = stats.namespaces
if TARGET_NAMESPACE in namespaces:
    uploaded_count = namespaces[TARGET_NAMESPACE].vector_count
    print(f"✅ SUCCESS: Namespace '{TARGET_NAMESPACE}' is active with {uploaded_count} vectors.")
else:
    print(f"❌ ERROR: Namespace '{TARGET_NAMESPACE}' not found.")
print("---------------------------------")

🧠 Loading BGE-Large Embedding Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


🚀 Starting upload to index 'medical-sport-assistant' under namespace 'agent_guide_Q&A'...
✅ Upload complete!

📊 --- PINECONE INDEX ANALYSIS ---
✅ SUCCESS: Namespace 'agent_guide_Q&A' is active with 5 vectors.
---------------------------------
